## 1. Extract all students that study Online, have an attendance_rate smaller than 50,
## and have more than 5 missed_deadlines. Store them in a dataframe called at_risk.

In [2]:
# import the libraries.
import pandas as pd
import numpy as np

In [3]:
# read the csv file and 5 rows of our data.
Student_df = pd.read_csv('/Users/chooseyourname/Desktop/student_dropout_risk_2026.csv')
Student_df.head()


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,...,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,...,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,...,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,4,7.4,1,0,7,0,10,24,5.8,0


In [4]:
# get access to all of the columns.
Student_df.columns

Index(['person_id', 'age', 'gender', 'nationality', 'first_generation',
       'disability_status', 'family_income_level', 'scholarship_holder',
       'tuition_up_to_date', 'part_time_job', 'weekly_work_hours',
       'course_type', 'study_mode', 'semester', 'international_student',
       'admission_grade', 'previous_qualification_grade', 'units_enrolled',
       'units_passed', 'units_failed', 'gpa_semester_1', 'gpa_semester_2',
       'attendance_rate', 'assignments_submitted_rate',
       'library_visits_per_month', 'lms_logins_per_week',
       'extracurricular_activities', 'missed_deadlines', 'stress_level',
       'mental_health_support', 'satisfaction_score', 'commute_time_mins',
       'sleep_hours', 'dropout'],
      dtype='object')

In [5]:
at_risk = Student_df[
    (Student_df['study_mode'] == 'Online') &
    (Student_df['attendance_rate'] < 50) &
    (Student_df['missed_deadlines'] > 5)
]
at_risk

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
478,479,22,Male,Germany,0,1,High,0,0,1,...,0,13.3,1,8,8,1,3,30,6.1,1
581,582,17,Female,India,1,0,Low,1,1,1,...,3,12.4,1,11,6,0,8,0,7.1,1
642,643,29,Male,Mexico,0,0,Low,1,1,1,...,4,14.3,0,9,10,0,7,9,4.0,1
1216,1217,15,Male,UK,0,0,Low,1,1,1,...,2,6.6,0,7,7,0,5,11,5.4,1
1316,1317,21,Male,Brazil,0,0,Low,0,1,1,...,3,7.8,0,17,5,0,4,6,8.1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96590,96591,23,Female,USA,1,0,Low,1,1,1,...,4,9.6,0,10,8,0,8,0,7.9,0
96648,96649,26,Female,USA,1,0,Low,1,1,1,...,2,12.4,0,8,9,1,6,10,6.5,1
97775,97776,19,Male,Germany,0,0,Low,1,0,1,...,6,7.5,1,8,6,1,10,19,8.8,1
99696,99697,26,Female,USA,1,0,Middle,1,0,1,...,2,14.3,1,16,9,1,8,12,7.8,1


## 2. Then compute, for each "course_type", how many of those students there are and what fraction of them has "dropped out". Store the result in a dataframe called "at_risk_by_course", with columns "students" and "dropout_rate", sorted by decreasing "dropout_rate".

In [7]:
at_risk_by_course = (
    at_risk.groupby('course_type').agg(students=('person_id', 'count'), 
    dropout_rate=('dropout', 'mean')).sort_values('dropout_rate', ascending=False)
)
at_risk_by_course

,students,dropout_rate
course_type,,
Law & Social Sciences,29,0.896552
Business,47,0.765957
STEM,47,0.765957
Arts & Humanities,40,0.750000
Health Sciences,34,0.647059


# 3. Add to the dataframe the following columns:

## pass_rate: the ratio between units_passed and units_enrolled;
## gpa_change: the difference between gpa_semester_2 and gpa_semester_1;
## gpa_trend: a string that is "improving" if gpa_change is larger than 0.5, "declining" if gpa_change is smaller than -0.5, and "stable" otherwise.

In [19]:
Student_df ['pass_rate'] = (Student_df['units_passed'] / Student_df['units_enrolled'])

Student_df ['gpa_change'] = (Student_df['gpa_semester_2'] - Student_df['gpa_semester_1'])

conditions = [
    Student_df['gpa_change'] > 0.5,
    Student_df['gpa_change'] < -0.5
]
choices = [
    'improving',
    'declining'
]

Student_df['gpa_trend'] = np.select(
    conditions, choices, default='stable'
)


Student_df[['units_passed', 'units_enrolled','pass_rate', 'gpa_change', 'gpa_trend']].head()

,units_passed,units_enrolled,pass_rate,gpa_change,gpa_trend
0,11,11,1.000000,-0.89,declining
1,11,14,0.785714,-0.03,stable
2,14,17,0.823529,-0.70,declining
3,11,13,0.846154,-0.41,stable
4,7,8,0.875000,-1.05,declining


## 4. Verify that, for every student, units_passed + units_failed equals units_enrolled; if not, report the rows violating this condition.



In [49]:
condition = (
    Student_df['units_passed'] + Student_df['units_failed']
    != Student_df['units_enrolled']
)
violations = Student_df.loc[condition]

violations


,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend


## 5. Compute the dropout rate for each value of gpa_trend, and the person_id of the 10 students with the largest GPA drop.

In [51]:
dropout_by_trend = (
    Student_df.groupby('gpa_trend')
    .agg(dropout_rate=('dropout', 'mean'))
)
dropout_by_trend

,dropout_rate
gpa_trend,
declining,0.265086
improving,0.589158
stable,0.391231


In [89]:
largest_gpa_drop = (
         Student_df.sort_values('gpa_change', ascending = True)
).head(10)
largest_gpa_drop[['person_id', 'gpa_change']]

,person_id,gpa_change
23225,23226,-2.49
96224,96225,-2.40
88489,88490,-2.37
51824,51825,-2.35
58935,58936,-2.33
46714,46715,-2.32
43035,43036,-2.30
72663,72664,-2.27
76705,76706,-2.23
23427,23428,-2.23


## 6. Partition the students into the following groups, according to weekly_work_hours: 0, 1-10, 11-20, 21-30, more than 30 (all intervals include both extremes).


In [114]:
conditions =[
(Student_df['weekly_work_hours'] == 0) ,
(Student_df['weekly_work_hours'] >= 1) & (Student_df['weekly_work_hours'] <= 10) ,
(Student_df['weekly_work_hours'] >= 11) & (Student_df['weekly_work_hours'] <= 20) ,
(Student_df['weekly_work_hours'] >= 21) & (Student_df['weekly_work_hours'] <= 30) ,
(Student_df['weekly_work_hours'] > 30)
]
choices = [
    '0 hours',
    '1-10 hours',
    '11-20 hours',
    '21-30 hours',
    'more than 30 hours'
]
Student_df['work_hours_group'] = np.select(
    conditions, choices, default='None'
)
#Student_df[['person_id', 'weekly_work_hours', 'work_hours_group']].head()

Student_df['work_hours_group'].value_counts()

work_hours_group
0 hours               51808
11-20 hours           23923
21-30 hours           15673
1-10 hours             6750
more than 30 hours     1846
Name: count, dtype: int64

# 7.Build a risk score by summing:
##    2 points if the tuition is not up to date;
##    1 point if stress_level is at least 8;
##    1 point if sleep_hours is smaller than 5;
##    1 point if the student has a part-time job with more than 20 weekly work hours;
##    2 points if attendance_rate is smaller than 60;
##    1 point if missed_deadlines is larger than 3.